### Needs `01-Prep-PipeA.ipynb` to be executed first

Reads from:
- `PipelineA_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Import Data

In [2]:
answers = pd.read_json("PipelineA_ynorm.json")

CATEGORIES = ["value", "unit"]
VARIANTS = ["A"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)
    
for col in pipeline_cols:
    answers[col] = answers[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [4]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [5]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [6]:
ynorm_exact, ynorm_exact_extended = eval(answers, True)

print_eval(ynorm_exact)

### value ###
A: True = 2094 || False = 114 || Quota = 94.84%

### unit ###
A: True = 1976 || False = 232 || Quota = 89.49%



In [7]:
ynorm_exact_extended.to_csv("PipelineA-Eval.csv")

In [8]:
ynorm_exact_extended.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   int64  
 5   scope            2208 non-null   str    
 6   page             490 non-null    float64
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_A          530 non-null    object 
 12  unit_A           530 non-null    object 
 13  label_A          530 non-null    object 
 14  A_value_hit      2208 non-null   bool   
 15  A_unit_hit       2208 non-null   bool   
dtypes: bool(2), float64(2), int64(1), object(5), str(6)
memory usage: 245.9

In [10]:
ynorm_exact_extended[ynorm_exact_extended["A_value_hit"]==False]

,report_name,status,scopes_present,years_present,year,scope,page,value,unit,unit_normalized,label,value_A,unit_A,label_A,A_value_hit,A_unit_hit
40,acuity brands inc_2022_report,complete,"[1, 2lb, 2mb, 3]",[2022],2021,3,NaN,NaN,NaN,NaN,NaN,[27651800.0],[MT CO2e],[Revised FY21 baseline - Use of Sold Products ...,False,False
142,allfunds group_2021_report,partial,"[1, 2lb]","[2019, 2020, 2021]",2019,2lb,142.0,436.43,tCO2e,t CO2e,Scope 2 CO 2 emissions,NaN,NaN,NaN,False,False
144,allfunds group_2021_report,partial,"[1, 2lb]","[2019, 2020, 2021]",2021,2lb,125.0,30.39,t CO2eq,t CO2e,Scope 2,NaN,NaN,NaN,False,False
145,allfunds group_2021_report,partial,"[1, 2lb]","[2019, 2020, 2021]",2021,2lb,142.0,30.39,tCO2e,t CO2e,Scope 2 CO 2 emissions,NaN,NaN,NaN,False,False
153,allfunds group_2021_report,partial,"[1, 2lb]","[2019, 2020, 2021]",2019,2mb,NaN,NaN,NaN,NaN,NaN,[436.43],[tCO2e],[Scope 2 CO2 emissions],False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2030,toshiba tec corp_2022_report,partial,"[1, 2lb]","[2020, 2021]",2021,2mb,NaN,NaN,NaN,NaN,NaN,[45.5],[thousand t-CO2],[Indirect emissions associated with energy use...,False,False
2039,toshiba tec corp_2022_report,partial,"[1, 2lb]","[2020, 2021]",2020,3,NaN,NaN,NaN,NaN,NaN,"[98.6, 42.5, 8.9, 2.3, 0.2, 1.6, 1.0, 275.5, 3.8]","[thousand t-CO2, thousand t-CO2, thousand t-CO...","[Category 1: Purchased goods and services, Cat...",False,False
2040,toshiba tec corp_2022_report,partial,"[1, 2lb]","[2020, 2021]",2021,3,NaN,NaN,NaN,NaN,NaN,"[102.3, 42.7, 10.7, 2.3, 0.2, 1.6, 0.8, 252.6,...","[thousand t-CO2, thousand t-CO2, thousand t-CO...","[Category 1: Purchased goods and services, Cat...",False,False
2061,uniper_2019_report,complete,"[1, 2lb, 2mb, 3]","[2017, 2018, 2019]",2019,2lb,16.0,1.12,million metric tons CO2,million t CO2,Scope 2 indirect emissions using the location-...,[1129277.0],[metric tons CO2],[Indirect Scope 2 emissions (location-based me...,False,False
